# 03 - Full-body detection (M4)
**Function in the system:** the detector locates the animal so the classifier sees the animal crop,
and it flags photos with no animal or several. Ground truth boxes are derived from the trimaps
(reviewed in notebook 01). The confidence and NMS thresholds are chosen on validation; the test
split is used once.

In [ ]:
# Setup — mount Drive before resolving the repository.
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

def _resolve_spv_repo(search_root: Path, explicit: str | None = None) -> Path:
    def valid(path):
        return (path / "src" / "spv").is_dir() and (path / "configs" / "config.yaml").is_file()
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not valid(candidate):
            raise FileNotFoundError(f"SPV_REPO is not a project root: {candidate}")
        return candidate
    candidates = set()
    for pattern in ("smart-pet-vision", "*smart-pet-vision", "*/smart-pet-vision", "*/*/smart-pet-vision"):
        candidates.update(path.resolve() for path in search_root.glob(pattern) if valid(path))
    if valid(search_root):
        candidates.add(search_root.resolve())
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one smart-pet-vision repository under {search_root}; found {len(candidates)}. "
            "Add the shared folder shortcut to My Drive, or set SPV_REPO to its exact path."
        )
    return candidates.pop()

if IN_COLAB:
    REPO = _resolve_spv_repo(Path("/content/drive/MyDrive"), os.environ.get("SPV_REPO"))
else:
    explicit = os.environ.get("SPV_REPO")
    local_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                       if (path / "src" / "spv").is_dir()), Path.cwd())
    REPO = _resolve_spv_repo(local_root, explicit)
os.chdir(REPO)
if IN_COLAB:
    !pip -q install -r requirements.txt
sys.path.insert(0, os.path.abspath("src"))

from spv.common import load_config, seed_everything, save_json, load_json
cfg = load_config()
if IN_COLAB:  # images on fast local disk (re-downloaded each session); results stay on Drive
    cfg["paths"]["raw"] = Path("/content/data/raw")
    cfg["paths"]["processed"] = Path("/content/data/processed")
seed_everything(cfg["seed"])
RUNS = Path(cfg["paths"]["runs"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from spv import data
df = data.ensure_data(cfg)          # first run: download + clean + split (about 10 min)
IMG = data.dataset_dir(cfg) / "images"
PROC = Path(cfg["paths"]["processed"])
print(len(df), "images after cleaning")

In [ ]:
from spv import detect
DET_YAML = PROC / "yolo_detect" / "data.yaml"
weights = detect.train(cfg, DET_YAML)
print(weights)

## Threshold sweep on validation

In [ ]:
OUT = RUNS / "detection"; OUT.mkdir(parents=True, exist_ok=True)
sweep = detect.threshold_sweep(cfg, weights, df[df.split == "val"], IMG)
sweep.to_csv(OUT / "threshold_sweep.csv", index=False)
op = detect.choose_operating_point(sweep, min_recall=0.80)
save_json({**op, "weights": os.path.relpath(weights)}, OUT / "operating_point.json"); op

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
for iou, g in sweep.groupby("nms_iou"):
    ax.plot(g.conf, g.precision, "o-", label=f"precision, NMS {iou}")
    ax.plot(g.conf, g.recall, "s--", label=f"recall, NMS {iou}")
ax.axvline(op["conf"], color="grey", lw=1); ax.set(xlabel="confidence threshold", title="Validation")
ax.legend(fontsize=7); plt.tight_layout(); fig.savefig(OUT / "figures_threshold_sweep.png", dpi=150)

## Test evaluation and galleries

In [ ]:
res = detect.run_test(cfg, weights, DET_YAML, df, IMG, op, OUT)
res["coco"], res["test_at_operating_point"], res["recall_by_size"]

In [ ]:
from IPython.display import Image as Show, display
display(Show(str(OUT / "figures" / "det_false_positives.jpg"), width=900))
display(Show(str(OUT / "figures" / "det_false_negatives.jpg"), width=900))

## Additional analysis (added 24 Oct 2026, no retraining)
These cells reuse the trained weights and the operating point chosen on validation above. They add what the
first run did not report: AP50-95 and precision/recall/F1 per class, a classified breakdown of every false
positive and false negative, a round-trip check of the exported YOLO labels with a contact sheet, the detector on
the team-captured photos (with recall by object size), and inference on two short videos. All figures: test
split or team set as stated, input 640 px, confidence 0.40 and NMS IoU 0.50 unless stated otherwise.

In [ ]:
import json
import pandas as pd
from spv import detect
OUT = RUNS / "detection"
op = load_json(OUT / "operating_point.json")
weights = Path(op["weights"])
IMGSZ = cfg["detection"]["imgsz"]
DET_YAML = PROC / "yolo_detect" / "data.yaml"
test_df = df[df.split == "test"]
gts = detect.gt_for(test_df)
preds = detect.predict_all(weights, [IMG / f"{n}.jpg" for n in test_df.image], conf=op["conf"],
                           iou=op["nms_iou"], imgsz=IMGSZ)
check = detect.score(preds, gts, op["conf"])
saved = load_json(OUT / "detection_test_metrics.json")["test_at_operating_point"]
assert (check["tp"], check["fp"], check["fn"]) == (saved["tp"], saved["fp"], saved["fn"]), (check, saved)
ap = detect.per_class_ap(weights, DET_YAML, IMGSZ)
per_class = detect.per_class_at_operating_point(preds, gts, op["conf"])
fp, fn = detect.error_breakdown(preds, gts, op["conf"])
fp.to_csv(OUT / "fp_breakdown.csv", index=False)
fn.to_csv(OUT / "fn_breakdown.csv", index=False)
both = sorted(set(fp.image) & set(fn.image))
extra = {"split": "test", "n_images": int(len(test_df)), "imgsz": IMGSZ, "conf": op["conf"],
         "nms_iou": op["nms_iou"], "tp_fp_fn": [check["tp"], check["fp"], check["fn"]],
         "per_class_ap": ap, "per_class_at_operating_point": per_class,
         "fp_by_kind": fp.kind.value_counts().to_dict(), "fn_by_kind": fn.kind.value_counts().to_dict(),
         "images_with_both_fp_and_fn": len(both)}
save_json(extra, OUT / "detection_extra_metrics.json")
print(json.dumps(extra, indent=1, default=float))

In [ ]:
cols = ["image", "pred", "gt", "conf", "best_iou"]
print("Class confusions among the false positives:")
print(fp[fp.kind.isin(["class_confusion", "localisation_and_class"])][cols].round(2).to_string(index=False))
print("\nMissed animals without any overlapping box:")
print(fn[fn.kind == "missed"].round(2).to_string(index=False))

In [ ]:
labels_root = PROC / "yolo_detect" / "labels"
rt = detect.label_roundtrip(df, labels_root)
roundtrip = {"labels": int(len(rt)), "missing": int((~rt.exists).sum()),
             "class_mismatch": int((~rt.class_ok).sum()), "max_abs_error_px": float(rt.max_err_px.max())}
save_json(roundtrip, OUT / "label_roundtrip.json")
print(roundtrip)
detect.label_sheet(df[df.split == "train"], labels_root, IMG, OUT / "figures" / "yolo_label_check.jpg")
from IPython.display import Image as Show, display
display(Show(str(OUT / "figures" / "yolo_label_check.jpg"), width=900))

In [ ]:
from PIL import Image as PILImage
TEAM = Path(cfg["paths"]["team"])
team_boxes = pd.read_csv(TEAM / "team_boxes.csv")
team_yaml = detect.export_team_yolo(team_boxes, TEAM / "images", PROC / "yolo_team")
team_ap = detect.per_class_ap(weights, team_yaml, IMGSZ)
tg = detect.team_gts(team_boxes)
tpred = detect.predict_all(weights, [TEAM / "images" / f"{n}.jpg" for n in tg], conf=op["conf"],
                           iou=op["nms_iou"], imgsz=IMGSZ)
team_op = detect.score(tpred, tg, op["conf"])
team_size = detect.recall_by_size_multi(tpred, tg, op["conf"])
sizes = {n: PILImage.open(TEAM / "images" / f"{n}.jpg").size for n in tg}
team_rel = detect.recall_by_relative_size(tpred, tg, sizes, op["conf"])
team_labels = pd.read_csv(TEAM / "labels.csv")
n_det = {k: int((v[2] >= op["conf"]).sum()) for k, v in tpred.items()}
cnt = team_labels.assign(n_detected=team_labels.image.map(n_det).fillna(0).astype(int))
team_det = {"images": int(len(tg)), "boxes": int(len(team_boxes)), "imgsz": IMGSZ, "conf": op["conf"],
            "nms_iou": op["nms_iou"], "coco": team_ap, "at_operating_point": team_op,
            "per_class_at_operating_point": detect.per_class_at_operating_point(tpred, tg, op["conf"]),
            "recall_by_size_coco_pixels": team_size.to_dict("records"),
            "recall_by_relative_size": team_rel.to_dict("records"),
            "images_count_correct": int((cnt.n_detected == cnt.n_animals).sum()),
            "multi_animal_images": int((cnt.n_animals > 1).sum()),
            "multi_animal_images_with_all_found": int(((cnt.n_animals > 1) & (cnt.n_detected >= cnt.n_animals)).sum()),
            "images_with_no_detection": int((cnt.n_detected == 0).sum())}
save_json(team_det, OUT / "team_detection_metrics.json")
print(json.dumps(team_det, indent=1, default=float))

In [ ]:
import re
import requests
VID = Path("/content/data/video") if Path("/content").exists() else Path("data/video")
VID.mkdir(parents=True, exist_ok=True)
clips = ["Cat in a Bag.webm", "Africanis Dogs Resting In Grass.ogv"]   # Wikimedia Commons, open licences
api = requests.get("https://commons.wikimedia.org/w/api.php", timeout=60,
                   headers={"User-Agent": "smart-pet-vision-capstone/1.0 (student project)"},
                   params={"action": "query", "format": "json", "prop": "imageinfo",
                           "iiprop": "url|extmetadata", "titles": "|".join("File:" + c for c in clips)}).json()
videos = []
for page in api["query"]["pages"].values():
    info = page["imageinfo"][0]
    meta = info["extmetadata"]
    path = VID / page["title"].removeprefix("File:").replace(" ", "_")
    if not path.exists():
        path.write_bytes(requests.get(info["url"], timeout=300,
                                      headers={"User-Agent": "smart-pet-vision-capstone/1.0"}).content)
    res = detect.video_inference(weights, path, OUT / "video" / f"{path.stem}_annotated.mp4", op["conf"],
                                 op["nms_iou"], IMGSZ, sheet_path=OUT / "figures" / f"video_{path.stem}.jpg")
    res.update(source=info["descriptionurl"] if "descriptionurl" in info else info["url"],
               licence=meta.get("LicenseShortName", {}).get("value"),
               author=re.sub("<[^>]+>", "", meta.get("Artist", {}).get("value", "")).strip())
    videos.append(res)
save_json(videos, OUT / "video_inference.json")
print(json.dumps(videos, indent=1, default=float))
for v in videos:
    display(Show(str(OUT / "figures" / f"video_{Path(v['video']).stem}.jpg"), width=900))

**Reading the results.** The operating point (confidence 0.40, NMS IoU 0.50) is the best validation F1 among
settings with recall >= 0.80: a missed animal sends a usable photo to staff, while a spurious box is usually caught by
the classifier-detector agreement check. The error breakdown above counts each false positive and false negative by
kind; the galleries show examples. The public test split has one large animal per image almost everywhere, so the
team-set evaluation in this section is the test for small animals and crowded scenes, and the videos show the
detector on moving footage (video is not part of the service).